# Build a candidate Gemma 3 `M_ft`

This is the canonical notebook for building a candidate Gemma 3-4B `M_ft` from frozen harmful-faces roles and recording face-sanity evidence. **A100 is the required runtime, not the experiment name.**

Start with seed 42. Face-sanity review decides only whether this is a candidate adapter worth retaining; it does not reproduce OOD emergent misalignment. Do not start primary RQ1 or BLOCK-EM until the separate reviewed OOD baseline exists for seeds 42, 43, and 44.

## 1. Confirm the required runtime

In Colab choose **Runtime → Change runtime type → GPU → A100** before continuing.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No CUDA GPU — enable a GPU runtime."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "A100" not in GPU_NAME:
    raise SystemExit(f"Refusing to continue on {GPU_NAME!r}. Switch the runtime to A100.")
assert torch.cuda.get_device_capability(0)[0] >= 8, "A100 bf16 capability is required."
print("A100 preflight passed.")

## 2. Set the seed and persistent Drive project

Drive is mandatory for this workflow. Data selection is frozen once at seed 42 and reused for every run; each training seed receives separate recovery checkpoints, result files, and W&B local artifacts.

In [ ]:
from pathlib import Path
import os

SEED = 42  # Training/LoRA randomness; change only here for 43 and 44 after seed 42 review.
DATA_SELECTION_SEED = 42  # Fixed shared faces split; never change for the three-seed replication.
HUB_NAMESPACE = "rlogger"  # Change only if your Hugging Face namespace differs.
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
for subdir in (
    "data", "data/splits", "checkpoints", "checkpoints/training",
    "results", "activations", "judge_cache", "runs", "wandb",
):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)

os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
os.environ["WANDB_DIR"] = str(DRIVE_PROJECT / "wandb")
os.environ["HF_HOME"] = "/content/hf-cache"  # model cache is ephemeral; artifacts are not

SPLIT_ROOT = DRIVE_PROJECT / "data" / "splits" / f"seed{DATA_SELECTION_SEED}"
TRAINING_DIR = DRIVE_PROJECT / "checkpoints" / "training" / f"FT_R32_gemma3_faces_seed{SEED}"
ADAPTER_DIR = DRIVE_PROJECT / "checkpoints" / f"FT_R32_gemma3_faces_seed{SEED}"
print("Drive project:", DRIVE_PROJECT)
print("Shared data-selection split root:", SPLIT_ROOT)
print("Recovery checkpoints:", TRAINING_DIR)


## 3. Get the versioned code

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
REPO_REF = "main"  # For a historical resume, set the exact original 40-char commit.

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise SystemExit(
            f"{REPO_DIR} exists but is not a git clone. Restart the runtime, then rerun."
        )
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise SystemExit(
            "The existing Colab clone has local changes. Restart the runtime, then rerun this cell; "
            "the notebook will not run stale source."
        )
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--prune", "--tags", "origin"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
checkout_ref = 'origin/main' if REPO_REF == 'main' else REPO_REF
subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", checkout_ref])

%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Repository commit:", REPO_COMMIT)
subprocess.check_call(["git", "-C", str(REPO_DIR), "status", "--short"])

## 4. Install the training runtime

Unsloth is installed before this project's broad dependency ranges. The fresh-process probe below is deliberate: it catches a CUDA/Torch/Unsloth mismatch before the expensive run. The exact runtime is persisted as a seed-specific manifest.

In [ ]:
from pathlib import Path
import importlib
import json
import subprocess
import sys

repo_dir = Path(globals().get("REPO_DIR", "/content/em-displacement-vlm")).resolve()
assert (repo_dir / "pyproject.toml").is_file(), "Run the clone cell first."
print("Python:", sys.executable)
print("Torch before install:")
subprocess.check_call([sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"])

constraints_path = repo_dir / "constraints" / "colab.txt"
assert constraints_path.is_file(), f"Missing constraints file: {constraints_path}"

# `pip install unsloth` is the official current installer; do not hard-code a
# wheel tag because Colab's Torch/CUDA pair changes. This CLI uses a fresh
# process after installation, so a notebook-kernel import cannot mask a mismatch.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "--constraint", str(constraints_path), "unsloth", "wandb==0.28.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(repo_dir), "--no-deps",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--constraint", str(constraints_path),
    "datasets>=2.19", "huggingface-hub>=0.23", "safetensors>=0.4",
    "pyyaml>=6.0", "trl",
])

print("Fresh-process training-runtime probe:")
fresh_runtime_output = subprocess.check_output([
    sys.executable, "-c",
    "import json, torch, unsloth; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'unsloth': 'OK'}))",
], text=True)
fresh_runtime = json.loads(fresh_runtime_output.strip().splitlines()[-1])
print(fresh_runtime)

repo_src = str(repo_dir / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()

import em_displacement_vlm
from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import checkpoint_dir, data_dir, results_dir

print("Project package:", Path(em_displacement_vlm.__file__).resolve())
for key, value in runtime_info().items():
    print(f"{key}: {value}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("results_dir:", results_dir())

from importlib.metadata import PackageNotFoundError, version

def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None

runtime_manifest = {
    "seed": SEED,
    "data_selection_seed": DATA_SELECTION_SEED,
    "git_commit": REPO_COMMIT,
    "python": sys.version,
    "gpu": GPU_NAME,
    "torch": fresh_runtime["torch"],
    "cuda": fresh_runtime["cuda"],
    "packages": {name: _package_version(name) for name in (
        "unsloth", "transformers", "trl", "peft", "datasets", "safetensors", "wandb"
    )},
    "pip_freeze": subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines(),
}
RUNTIME_MANIFEST = DRIVE_PROJECT / "runs" / f"environment_ft_seed{SEED}.json"
serialized_runtime_manifest = json.dumps(runtime_manifest, indent=2, sort_keys=True) + "\n"
if RUNTIME_MANIFEST.exists() and RUNTIME_MANIFEST.read_text() != serialized_runtime_manifest:
    raise SystemExit(f"Runtime differs from frozen manifest: {RUNTIME_MANIFEST}")
if not RUNTIME_MANIFEST.exists():
    RUNTIME_MANIFEST.write_text(serialized_runtime_manifest)
print("Runtime manifest:", RUNTIME_MANIFEST)

## 5. Authenticate model access and experiment tracking

Create/select a **private** W&B project named `em-displacement-vlm` before running this cell. The held-out sanity table logs prompts and generated responses, but not images. Add fresh Colab secrets named `HF_TOKEN` and `WANDB_API_KEY`; no GitHub token is needed.

In [ ]:
from google.colab import userdata
import os

WANDB_ENABLED = True
WANDB_PROJECT = "em-displacement-vlm"
WANDB_ENTITY = None  # Set a private team/entity slug only if needed.

def _set_secret(name: str, *, required: bool) -> None:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        if required:
            raise SystemExit(f"Secret not set: {name}")
        return
    os.environ[name] = value
    print(f"Loaded secret: {name}")

_set_secret("HF_TOKEN", required=True)
_set_secret("WANDB_API_KEY", required=WANDB_ENABLED)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

if WANDB_ENABLED:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    import wandb
    wandb.login(verify=True)
    # wandb.login starts a shared wandb-core and exports WANDB_SERVICE into this
    # kernel. FT/sanity run in subprocesses; if they inherit that handle they hit
    # "run ID … is in use". Drop it so each script owns its own service.
    os.environ.pop("WANDB_SERVICE", None)
    print(f"W&B tracking ready for project: {WANDB_PROJECT}")

## 6. Verify Gemma 3-4B model access

This checks the exact pinned revision before the long fine-tune.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="unsloth/gemma-3-4b-it",
    filename="config.json",
    revision="bf46152c47f5dd20b896357cb51abc4c03b8ee8c",
    token=True,
)
print("Pinned Gemma access confirmed:", path)

## 7. Freeze this seed's hash-disjoint data roles

The role root is immutable after creation. Rerunning this cell reuses and verifies the existing seed instead of overwriting it.

In [ ]:
import json
import subprocess
import sys

SPLIT_MANIFEST = SPLIT_ROOT / "manifest.json"
if SPLIT_MANIFEST.is_file():
    frozen_manifest = json.loads(SPLIT_MANIFEST.read_text())
    if frozen_manifest.get("seed") != DATA_SELECTION_SEED or frozen_manifest.get("mode") != "hf":
        raise SystemExit(f"Existing split root does not identify HF data-selection seed {DATA_SELECTION_SEED}: {SPLIT_ROOT}")
    print("Reusing shared frozen data-selection role:", SPLIT_ROOT)
elif SPLIT_ROOT.exists() and any(SPLIT_ROOT.iterdir()):
    raise SystemExit(
        f"Incomplete split root: {SPLIT_ROOT}. Do not overwrite it; choose a fresh root."
    )
else:
    subprocess.check_call([
        sys.executable, "scripts/prepare_datasets.py", "--use-hf",
        "--seed", str(DATA_SELECTION_SEED), "--out", str(SPLIT_ROOT),
    ])

subprocess.check_call([sys.executable, "scripts/check_disjointness.py", "--root", str(SPLIT_ROOT)])

## 8. Materialize this seed's immutable run configuration

This config points FT at the shared immutable Drive split and this training seed's directory. It saves three full recovery checkpoints, every 25 updates, and resumes only the newest valid checkpoint from this exact run.

In [ ]:
from pathlib import Path
import yaml

HUB_REPO = f"{HUB_NAMESPACE}/FT_R32_gemma3_faces_seed{SEED}"
base_cfg_path = Path("configs/reproduce_mft_gemma3.yaml")
cfg = yaml.safe_load(base_cfg_path.read_text())
cfg.update({
    "hub_repo": HUB_REPO,
    "seed": SEED,
    "data_selection_seed": DATA_SELECTION_SEED,
    "run_name": f"reproduce_mft_gemma3_r32_seed{SEED}",
    "output_dir": str(TRAINING_DIR),
    "split_root": str(SPLIT_ROOT),
    "save_steps": 25,
    "save_total_limit": 3,
    "resume_from_checkpoint": "auto",
    "push_to_hub": False,
    "use_wandb": WANDB_ENABLED,
    "wandb_project": WANDB_PROJECT,
    "wandb_entity": WANDB_ENTITY,
    "wandb_group": "mft-gemma3-r32",
})
RUN_CONFIG = DRIVE_PROJECT / "runs" / f"reproduce_mft_gemma3_r32_seed{SEED}.yaml"
rendered_run_config = yaml.safe_dump(cfg, sort_keys=False)
if RUN_CONFIG.exists():
    if RUN_CONFIG.read_text() != rendered_run_config:
        raise SystemExit(
            f"Existing run config differs: {RUN_CONFIG}. Do not overwrite a seed run."
        )
    print("Reusing materialized run config:", RUN_CONFIG)
else:
    RUN_CONFIG.write_text(rendered_run_config)
    print("Created materialized run config:", RUN_CONFIG)
print(RUN_CONFIG.read_text())

## 9. Fine-tune Gemma 3-4B → `M_ft`

Recovery checkpoints live in `TRAINING_DIR`, not in the final adapter directory. If Colab interrupts, rerun this cell **without changing the checked-out commit, runtime manifest, materialized config, or split** and it resumes the newest valid checkpoint plus the same W&B run. For a historical checkpoint, restart the runtime, set `REPO_REF` in step 3 to the exact commit recorded by that run, and use that commit's notebook, constraints, materialized config, split, and saved runtime manifest. If any of those are missing or disagree, archive the historical run and start a fresh current-protocol run. Never relabel a mixed-code resume as one reproducible run.

For an intentional **from-scratch** restart of this seed (no valid `checkpoint-*`, or a new protocol commit that must not mix with the old Drive identity), set `RESET_TRAINING_DIR = True` in the next cell once. That archives `TRAINING_DIR` aside and leaves the shared `data/splits/seed42` role untouched.

In [ ]:
from datetime import datetime, timezone
import shutil

# Set True once for an intentional from-scratch restart of THIS seed's training dir.
# Leaves the shared data/splits/seed42 role and final adapter dir alone.
RESET_TRAINING_DIR = False

if RESET_TRAINING_DIR:
    if not TRAINING_DIR.exists():
        print("No training dir to archive:", TRAINING_DIR)
    else:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archived = TRAINING_DIR.with_name(f"{TRAINING_DIR.name}.archived-{stamp}")
        if archived.exists():
            raise SystemExit(f"Archive target already exists: {archived}")
        shutil.move(str(TRAINING_DIR), str(archived))
        TRAINING_DIR.mkdir(parents=True, exist_ok=True)
        print("Archived previous training dir to:", archived)
        print("Fresh training dir ready:", TRAINING_DIR)
else:
    print("Keeping TRAINING_DIR as-is:", TRAINING_DIR)
    print("Set RESET_TRAINING_DIR = True above only for a deliberate from-scratch restart.")

In [ ]:
import os
import subprocess
import sys

env = dict(os.environ)
env.pop("WANDB_SERVICE", None)

result = subprocess.run(
    [sys.executable, "-u", "scripts/ft_faces.py", "--config", str(RUN_CONFIG)],
    env=env,
    check=False,
)
if result.returncode:
    raise SystemExit(
        f"ft_faces.py failed with exit code {result.returncode}. "
        "Scroll up for the script traceback (unbuffered via -u)."
    )
print("FT subprocess finished successfully.")

## 10. Generate held-out sanity evidence

The sanity job uses the same shared frozen role. It produces a core image probe, a text-only bleed-through probe, and a held-out batch. W&B receives prompts and responses only—no images. After a completed FT in a fresh runtime, run sections 1–5 and then section 10; do not rerun sections 7–9.

In [ ]:
import json
import yaml

assert ADAPTER_DIR.exists(), f"Missing completed adapter: {ADAPTER_DIR}"

sanity_cfg = yaml.safe_load(Path("configs/sanity_em.yaml").read_text())
sanity_cfg.update({
    "model_id": str(ADAPTER_DIR),
    "seed": SEED,
    "data_selection_seed": DATA_SELECTION_SEED,
    "run_name": f"verify_mft_gemma3_seed{SEED}_bf16",
    "split_root": str(SPLIT_ROOT),
    "load_in_4bit": False,
    "use_wandb": WANDB_ENABLED,
    "wandb_project": WANDB_PROJECT,
    "wandb_entity": WANDB_ENTITY,
    "wandb_group": "mft-gemma3-r32",
})
SANITY_CONFIG = DRIVE_PROJECT / "runs" / f"verify_mft_gemma3_seed{SEED}_bf16.yaml"
rendered_sanity_config = yaml.safe_dump(sanity_cfg, sort_keys=False)
if SANITY_CONFIG.exists():
    if SANITY_CONFIG.read_text() != rendered_sanity_config:
        raise SystemExit(
            f"Existing sanity config differs: {SANITY_CONFIG}. Do not overwrite a seed run."
        )
    print("Reusing materialized sanity config:", SANITY_CONFIG)
else:
    SANITY_CONFIG.write_text(rendered_sanity_config)
    print("Created materialized sanity config:", SANITY_CONFIG)

loader_path = REPO_DIR / "src" / "em_displacement_vlm" / "evals" / "sanity_em.py"
assert loader_path.is_file(), f"Missing sanity loader: {loader_path}"
adapter_config_path = ADAPTER_DIR / "adapter_config.json"
assert adapter_config_path.is_file(), f"Missing adapter metadata: {adapter_config_path}"
stored_base = str(json.loads(adapter_config_path.read_text()).get("base_model_name_or_path") or "")
if "-unsloth-bnb-" in stored_base:
    print(
        "Detected Unsloth's internal quantization marker in the adapter. "
        f"Sanity will load the pinned base: {sanity_cfg['base_model_id']}"
    )
else:
    print(f"Adapter base metadata: {stored_base or sanity_cfg['base_model_id']}")

### Run the sanity check

This reads the completed adapter from Drive and writes new held-out evidence. It does not fine-tune or overwrite the adapter.

In [ ]:
import os
import subprocess
import sys

env = dict(os.environ)
env.pop("WANDB_SERVICE", None)

result = subprocess.run(
    [sys.executable, "-u", "scripts/sanity_check_em.py", "--config", str(SANITY_CONFIG)],
    env=env,
    check=False,
)
if result.returncode:
    raise SystemExit(
        f"sanity_check_em.py failed with exit code {result.returncode}. "
        "Scroll up for the script traceback (unbuffered via -u)."
    )
print("Sanity subprocess finished successfully.")


## 11. Candidate-adapter handoff

Read all three outputs. The scripts do not infer a scientific result from response length or generation count. This notebook records face-sanity evidence only; complete the matched base/FT review in notebook 02 before deciding whether to retain or privately publish a candidate adapter.

In [ ]:
print(
    "FT and face-sanity evidence are saved to Drive. This notebook cannot certify EM, "
    "set a behavioral decision, or upload an adapter. Continue in notebook 02."
)

## 12. Continue in notebook 02

Notebook 02 generates the matched base evidence, creates the blinded candidate-adapter review, records an explicit decision, and offers a protected private upload that requires the review summary. Keep Drive recovery checkpoints private; they are for resuming FT, not for publication.

In [ ]:
print("Next notebook: 02_review_candidate_adapter.ipynb")

## 13. Build the remaining candidate adapters

For seeds 43 and 44, change only `SEED` in section 2, then run **sections 7 → 12 in order**: verify the shared frozen role → materialize config → FT → face-sanity evidence → notebook 02 candidate review. All three runs reuse `data/splits/seed42`; each retains its own training directory, adapter, configs, runtime manifest, and W&B run. Primary RQ1 remains blocked until the separate OOD paper-comparable baseline is reviewed across all three seeds.